In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
#from utils import *
#from utils_cpdir import *
from utils_mvcp import *
from utils import *
from utils_spec import *
import os
from tqdm import tqdm
import json
from datetime import date

In [2]:
# Get the current date
todays_date = date.today()
year = str(todays_date.year)
if todays_date.month < 10:
    month = '0'+str(todays_date.month)
else:
    month = str(todays_date.month)

if todays_date.day < 10:
    day = '0'+str(todays_date.day)
else:
    day = str(todays_date.day)

current_date = year+month+day
directory_name = 'results/'+current_date
if not os.path.isdir(directory_name):
    os.mkdir(directory_name)
    
directory_name

'results/20251110'

In [3]:
#directory_name = 'results/20251026'
#directory_name

In [4]:
remove_slopp = pd.read_excel('RochmanSpectra_ToRemove.xlsx', sheet_name="SLOPPe Remove")

mIRageLibraySpectra_Removed = pd.read_csv('../gloves_contamination/mIRageLibraySpectra_Removed.csv')
mIRageLibraySpectra_Removed.iloc[-1,2] = 'PE_15'

dict_slopp_reb = {'PE':'Polyethylene', 
                  'PP':'Polypropylene', 
                  'PSL':'Polystyrene'}

with open('processed_data/passive_data_raman_saturation.json', 'r') as f:
    saturated_wn_dict = json.load(f)

    
ids_sat = list(saturated_wn_dict.keys())

slopp_df = pd.read_csv('processed_data/slopp_data_raman_final_specy.csv')
del slopp_df['Unnamed: 0']
slopp_df.columns = list(np.float64(slopp_df.columns[:-4])) + list(slopp_df.columns[-4:])

slopp_df = slopp_df.query('polymer != "Cotton" and polymer !="Dyed Cellulose"').reset_index(drop=True)
slopp_df.loc[slopp_df['polymer'] == "Polyester",'polymer'] = "Polyethylene Terephthalate"
slopp_df = slopp_df.loc[[x not in np.array(remove_slopp['Example']) for x in slopp_df['id']], :].reset_index(drop=True)

rebecca_raman_df = pd.read_csv('processed_data/rebecca_data_raman_final_specy.csv')
del rebecca_raman_df['Unnamed: 0']
rebecca_raman_df.columns = list(np.float64(rebecca_raman_df.columns[:-4])) + list(rebecca_raman_df.columns[-4:])

rebecca_raman_df = rebecca_raman_df.drop(np.where([x in list(mIRageLibraySpectra_Removed.query('Signal == "Raman"')['ID']) for x in rebecca_raman_df['id']])[0],
                             axis=0).reset_index(drop=True)

rebecca_raman_df = rebecca_raman_df.query('polymer in @dict_slopp_reb').reset_index(drop=True)
rebecca_raman_df['polymer'] = [dict_slopp_reb[x] for x in rebecca_raman_df['polymer']]

passive_data_raman = pd.read_csv('processed_data/passive_data_raman_specy.csv')
del passive_data_raman['Unnamed: 0']
del passive_data_raman['wn_saturated']
del passive_data_raman['id_file']
ids_no_signal = np.array(passive_data_raman.query('no_signal>0')['id'])
del passive_data_raman['no_signal']
passive_data_raman.columns = list(np.float64(passive_data_raman.columns[:-3])) + list(passive_data_raman.columns[-3:])
passive_data_raman['polymer'] = ''
passive_data_raman['label'] = ''


#1850-2750

wn_raman = np.float64(slopp_df.columns[:-4])

min_raman_1 = 600
max_raman_1 = 1000

min_raman_2 = 1600
max_raman_2 = 3500

idx_1 = list(np.logical_or(
    np.logical_and(
        wn_raman > min_raman_1,
        wn_raman < max_raman_1),
    np.logical_and(
        wn_raman > min_raman_2,
        wn_raman < max_raman_2))) + [True]*4

min_raman_1 = 600
max_raman_1 = 910

min_raman_2 = 1040
max_raman_2 = 2310

min_raman_3 = 2500
max_raman_3 = 3500

#min_raman_1 = 600
#max_raman_1 = 1000

#min_raman_2 = 1600
#max_raman_2 = 3500

#min_raman_3 = 2500
#max_raman_3 = 3500

idx_1 = list(
    np.logical_or( 
        np.logical_or(
            np.logical_and(
                wn_raman > min_raman_1,
                wn_raman < max_raman_1),
            np.logical_and(
                wn_raman > min_raman_2,
                wn_raman < max_raman_2)
            ),   
        np.logical_and(
            wn_raman > min_raman_3,
            wn_raman < max_raman_3)
    )
    ) + [True]*4

idx_2 = idx_1[:-4] + [False]*4

slopp_df = slopp_df.loc[:,idx_1]
rebecca_raman_df = rebecca_raman_df.loc[:,idx_1]
passive_data_raman = passive_data_raman.loc[:,idx_1+[True]]
#passive_data_raman = passive_data_raman.query('id != "DC_passive_June2024_I7_20240717_90"').reset_index(drop=True)

slopp_df.iloc[:,:-4] = (slopp_df.iloc[:,:-4] - slopp_df.iloc[:,:-4].mean(1).to_numpy()[:,None]) / slopp_df.iloc[:,:-4].std(1).to_numpy()[:,None]
rebecca_raman_df.iloc[:,:-4] = (rebecca_raman_df.iloc[:,:-4] - rebecca_raman_df.iloc[:,:-4].mean(1).to_numpy()[:,None]) / rebecca_raman_df.iloc[:,:-4].std(1).to_numpy()[:,None]
passive_data_raman.iloc[:,:-5] = (passive_data_raman.iloc[:,:-5] - passive_data_raman.iloc[:,:-5].mean(1).to_numpy()[:,None]) / passive_data_raman.iloc[:,:-5].std(1).to_numpy()[:,None]

ids_no_signal = np.unique(
    np.concatenate([ids_no_signal,
    np.array(passive_data_raman.loc[passive_data_raman.iloc[:,:-5].T.std()<.01,'id'])]))

idx_raman_nan = np.unique(np.where(np.isnan(passive_data_raman.iloc[:,:-5]))[0])


In [5]:
#slopp_df.to_csv(directory_name+'/slopp_data_raman_final_specy.csv')

In [6]:
#passive_data_raman.to_csv(directory_name+'/passive_data_raman_specy.csv')

In [7]:
#label_table_ir.to_csv(directory_name+'/label_table_ir.csv')

In [8]:
#passive_data_raman['saturation'] = [x in ids_sat for x in passive_data_raman['id']]
#passive_data_raman['no_signal'] = [x in ids_no_signal for x in passive_data_raman['id']]
#passive_data_raman.to_csv(directory_name+'/passive_data_raman_specy.csv')

In [9]:
#slopp_df.to_csv(directory_name+'/slopp_data_raman_final_specy.csv')

In [10]:
remove_flopp = pd.read_excel('RochmanSpectra_ToRemove.xlsx', sheet_name="FLOPPe Remove")

flopp_df = pd.read_csv('processed_data/flopp_data_ir_final.csv')
del flopp_df['Unnamed: 0']
flopp_df.columns = list(np.float64(flopp_df.columns[:-4])) + list(flopp_df.columns[-4:])
flopp_df = flopp_df.query('polymer != "Cotton"').reset_index(drop=True)
flopp_df.columns = list(np.float64(flopp_df.columns[:-4])) + list(flopp_df.columns[-4:])
flopp_df = flopp_df.loc[[x not in np.array(remove_flopp['Example']) for x in flopp_df['id']], :].reset_index(drop=True)

rebecca_ir_df = pd.read_csv('processed_data/rebecca_data_ir_final.csv')
del rebecca_ir_df['Unnamed: 0']
rebecca_ir_df.columns = list(np.float64(rebecca_ir_df.columns[:-4])) + list(rebecca_ir_df.columns[-4:])

rebecca_ir_df = rebecca_ir_df.drop(np.where([x in list(mIRageLibraySpectra_Removed.query('Signal == "PTIR"')['ID']) for x in rebecca_ir_df['id']])[0],
                             axis=0).reset_index(drop=True)

rebecca_ir_df = rebecca_ir_df.query('polymer in @dict_slopp_reb').reset_index(drop=True)
rebecca_ir_df['polymer'] = [dict_slopp_reb[x] for x in rebecca_ir_df['polymer']]


passive_data_ir = pd.read_csv('processed_data/passive_data_ir.csv')
del passive_data_ir['Unnamed: 0']
background_ids = np.array(passive_data_ir.query('background == "background"')['id'])
passive_data_ir.columns = list(np.float64(passive_data_ir.columns[:-4])) + list(passive_data_ir.columns[-4:])
passive_data_ir['polymer'] = ''
passive_data_ir['label'] = ''

#passive_data_ir = passive_data_ir.query('id != "DC_passive_June2024_I7_20240717_90"').reset_index(drop=True)

flopp_df.iloc[:,:-4] = (flopp_df.iloc[:,:-4] - flopp_df.iloc[:,:-4].mean(1).to_numpy()[:,None]) / flopp_df.iloc[:,:-4].std(1).to_numpy()[:,None]
rebecca_ir_df.iloc[:,:-4] = (rebecca_ir_df.iloc[:,:-4] - rebecca_ir_df.iloc[:,:-4].mean(1).to_numpy()[:,None]) / rebecca_ir_df.iloc[:,:-4].std(1).to_numpy()[:,None]
passive_data_ir.iloc[:,:-6] = (passive_data_ir.iloc[:,:-6] - passive_data_ir.iloc[:,:-6].mean(1).to_numpy()[:,None]) / passive_data_ir.iloc[:,:-6].std(1).to_numpy()[:,None]

In [11]:
#passive_data_ir.to_csv(directory_name+'/passive_data_ir.csv')

In [12]:
#particle_info = pd.read_csv('processed_data/passive_data_raman_specy_particle_info.csv')
#particle_info = particle_info.loc[:,['id','size','area','perimeter','circularity','distance']]
#particle_info.to_csv('processed_data/particle_info_passive_data.csv')

In [14]:
ids_rubber = pd.read_excel('evaluation_data/Labeled_D&G_RamanOnly_Passive.xlsx', header=None)[0].to_numpy()
rubber_data = passive_data_raman.query('id in @ids_rubber').reset_index(drop=True)#.iloc[:,:-5]
rubber_data['polymer'] = 'Rubber'
rubber_data = rubber_data.drop('label',axis=1)
rubber_data = rubber_data.iloc[:,list(range(rubber_data.shape[1]-2))+[-1,-2]]
passive_data_raman_rest = passive_data_raman.query('id not in @ids_rubber').reset_index(drop=True)

In [36]:
rubber_data.to_csv(directory_name+'/passive_uncertain_rubber_data_raman_specy.csv')

In [15]:
#passive_data_ir_paired = passive_data_ir.query('id not in @ids_sat and id not in @background_ids and id not in @ids_no_signal').reset_index(drop=True)
#passive_data_ir_unpaired_ram = passive_data_ir.query('(id not in @ids_sat and id not in @ids_no_signal) and id in @background_ids').reset_index(drop=True)
#passive_data_ir_unpaired_ir = passive_data_ir.query('(id in @ids_sat or id in @ids_no_signal) and id not in @background_ids').reset_index(drop=True)
#passive_data_ir_nopair = passive_data_ir.query('(id in @ids_sat or id in @ids_no_signal) and id in @background_ids').reset_index(drop=True)

passive_data_raman_paired = passive_data_raman_rest.query('id not in @idx_raman_nan and (id not in @ids_sat and id not in @ids_no_signal) and id not in @background_ids').reset_index(drop=True)
passive_data_raman_unpaired_ram = passive_data_raman_rest.query('id not in @idx_raman_nan and (id not in @ids_sat and id not in @ids_no_signal) and id in @background_ids').reset_index(drop=True)
passive_data_raman_unpaired_ir = passive_data_raman_rest.query('(id not in @idx_raman_nan or id in @ids_sat or id in @ids_no_signal) and id not in @background_ids').reset_index(drop=True)
passive_data_raman_nopair = passive_data_raman_rest.query('(id not in @idx_raman_nan or id in @ids_sat or id in @ids_no_signal) and id in @background_ids').reset_index(drop=True)


In [16]:
print('unpaired Raman spectra: ', passive_data_raman_unpaired_ram.shape[0])
print('unpaired IR spectra: ', passive_data_raman_unpaired_ir.shape[0])
print('no pair spectra: ', passive_data_raman_nopair.shape[0])
print('paired spectra: ', passive_data_raman_paired.shape[0])

unpaired Raman spectra:  1713
unpaired IR spectra:  6169
no pair spectra:  1906
paired spectra:  4073


In [17]:
np.random.seed(0)
idx = np.random.rand(rubber_data.shape[0]) > .50
rubber_ref = rubber_data.loc[idx,:].reset_index(drop=True)
rubber_cal = rubber_data.loc[~idx,:].reset_index(drop=True)

In [18]:
flopp_slopp_labels = np.array(slopp_df.groupby('polymer').count().iloc[:,-1:].merge(flopp_df.groupby('polymer').count().iloc[:,-1:], 
                                                      left_index=True,
                                                      right_index=True, how='inner').index)

flopp_df = flopp_df.query('polymer in @flopp_slopp_labels').reset_index(drop=True)
slopp_df = slopp_df.query('polymer in @flopp_slopp_labels').reset_index(drop=True)


In [19]:
idx_both_rebecca = np.array(rebecca_ir_df.loc[[x in np.array(rebecca_raman_df['id']) for x in rebecca_ir_df['id']],'id'])# 
rebecca_ir_df = rebecca_ir_df.query('id in @idx_both_rebecca').reset_index(drop=True)
rebecca_raman_df = rebecca_raman_df.query('id in @idx_both_rebecca').reset_index(drop=True)


# Paired

In [24]:
(reference_library_raman, 
 calibration_set_raman, 
 test_set_raman, 
 label_table_raman) = gen_data_sets(rubber_ref,#.loc[:,idx_1], 
                                    rubber_cal,#.loc[:,idx_1], 
                                    passive_data_raman_paired)#.loc[:,idx_1+[True]])

(X_train_raman, X_cal_raman, X_test_raman, 
 y_train_raman, y_cal_raman, y_test_raman, 
 label_train_raman, label_cal_raman, label_test_raman, 
 n_test_raman, wl_list_raman) = process_data(reference_library_raman,
                                             calibration_set_raman,
                                             test_set_raman)




In [25]:
alpha = 0.1

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

In [27]:
def smx_log(X_train, X_test, labels_train, C=0.01, return_weights=False):
    
    clf = LogisticRegression(random_state=0, C=C).fit(X_train, labels_train)
    smx = clf.predict_proba(X_test)
    label_pred = clf.predict(X_test)
    if return_weights:
        return smx, clf
    
    return smx

In [28]:
def smx_lda(X_train, X_test, labels_train, fact=1):
    clf = LinearDiscriminantAnalysis(store_covariance=True)
    clf.fit(X_train, labels_train)
    scores = (clf.decision_function(X_test) -
              #fact*1/2*(X_test[:,None,:]@np.linalg.pinv(clf.covariance_)@X_test[:,:,None])[:,:,0]
     fact*1/2*(X_test[:,None,:]@np.linalg.inv(clf.covariance_+0.01*np.identity(clf.covariance_.shape[0]))@X_test[:,:,None])[:,:,0]
             )
    #scipy.linalg.pinv
    return scores

In [29]:
# Calculate HQI for calibration set
smx_cal_raman = smx_pcc(X_train_raman, 
                  X_cal_raman, 
                  label_train_raman, 
                      #C=0.002
                      )

# Calculate HQI for test set
smx_test_raman = smx_pcc(X_train_raman, 
                   X_test_raman, 
                   label_train_raman, 
                        #C=0.002
                       )

#smx_test_all = smx_nn_all(
#        X_train_raman, 
#        X_test_raman, 
#        label_train_raman
#)

#pred_slopp_df = [
#    np.array(label_table_raman.iloc[np.argmax(smx_test, 1),1]),
#    np.max(smx_test, 1),
#    np.array(reference_library_raman.iloc[np.argmax(smx_test_all, 0),0])
#]

# Set confidence level 1 - alpha
n = smx_cal_raman.shape[0]
#for alpha in [alpha]:
prediction_sets_raman, qhat_raman = conform_pred(smx_cal_raman, smx_test_raman, label_cal_raman, n, alpha, cond=0)

prediction_sets_n_raman = [list(label_table_raman.loc[ps,'polymer']) for ps in prediction_sets_raman]
prediction_sets_size_raman = np.array([len(ps) for ps in prediction_sets_n_raman])

zero_label_raman = np.where(prediction_sets_size_raman == 0)[0]
single_label_raman = np.where(prediction_sets_size_raman == 1)[0]
mult_label_raman = np.where(prediction_sets_size_raman > 1)[0]

print('Sets with zero labels: ' + str(zero_label_raman.shape[0]))
print('Sets with one label: ' + str(single_label_raman.shape[0]))
print('Sets with >1 labels: ' + str(mult_label_raman.shape[0]))
print('Mean set size:', prediction_sets_raman.sum(1).mean())
print('Std set size:', prediction_sets_raman.sum(1).std())
print('HQI cutt off:', 1-qhat_raman)

Sets with zero labels: 3146
Sets with one label: 927
Sets with >1 labels: 0
Mean set size: 0.22759636631475572
Std set size: 0.4192806462920453
HQI cutt off: 0.9577787447942816


In [30]:
output = [prediction_sets_n_raman[i][0] if i in single_label_raman else 
              '--outlier--' if i in zero_label_raman else 
              '--manual check!--' for i in range(label_test_raman.shape[0])]
output_df_raman = pd.DataFrame({'id':test_set_raman['id'],
                          'prection_set_raman':prediction_sets_n_raman,
                              #'output':output
                              })
output_df_raman

,id,prection_set_raman
0,FHS_passive_August2024_B4_20240910_1,[]
1,FHS_passive_August2024_B4_20240910_2,[]
2,FHS_passive_August2024_B4_20240910_3,[]
3,FHS_passive_August2024_B4_20240910_4,[]
4,FHS_passive_August2024_B4_20240910_5,[]
...,...,...
4068,IAO_passive_June2024_G7_20240801_39,[]
4069,IAO_passive_June2024_G7_20240801_40,[]
4070,IAO_passive_June2024_G7_20240801_41,[]
4071,IAO_passive_June2024_G7_20240801_42,[]


In [31]:
#Passive_MVCP_Review = pd.read_csv('evaluation_data/Passive_MVCP_Review.csv')['ID'].to_numpy().flatten()
output_df_raman.to_csv(directory_name+'/CP_Rubber_results_passive_paired.csv')



# Unpaired Raman

In [32]:
(reference_library_raman, 
 calibration_set_raman, 
 test_set_raman, 
 label_table_raman) = gen_data_sets(rubber_ref, 
                              rubber_cal, 
                              passive_data_raman_unpaired_ram)

(X_train_raman, X_cal_raman, X_test_raman, 
 y_train_raman, y_cal_raman, y_test_raman, 
 label_train_raman, label_cal_raman, label_test_raman, 
 n_test_raman, wl_list_raman) = process_data(reference_library_raman,
                                 calibration_set_raman,
                                 test_set_raman)

In [33]:
# Calculate HQI for calibration set
smx_cal = smx_pcc(X_train_raman, 
                 X_cal_raman, 
                 label_train_raman, 
                 # C=0.002
                 )

# Calculate HQI for test set
smx_test = smx_pcc(X_train_raman, 
                  X_test_raman, 
                  label_train_raman, 
                  # C=0.002
                  )

#smx_test_all = smx_nn_all(
#        X_train_raman, 
#        X_test_raman, 
#        label_train_raman
#)

#pred_slopp_df = [
#    np.array(label_table_raman.iloc[np.argmax(smx_test, 1),1]),
#    np.max(smx_test, 1),
#    np.array(reference_library_raman.iloc[np.argmax(smx_test_all, 0),0])
#]

# Set confidence level 1 - alpha
n = smx_cal.shape[0]
alpha = 0.1
#for alpha in [alpha]:
prediction_sets_raman, qhat_raman = conform_pred(smx_cal, smx_test, label_cal_raman, n, alpha, cond=0)

prediction_sets_n_raman = [list(label_table_raman.loc[ps,'polymer']) for ps in prediction_sets_raman]
prediction_sets_size_raman = np.array([len(ps) for ps in prediction_sets_n_raman])

zero_label_raman = np.where(prediction_sets_size_raman == 0)[0]
single_label_raman = np.where(prediction_sets_size_raman == 1)[0]
mult_label_raman = np.where(prediction_sets_size_raman > 1)[0]

print('Sets with zero labels: ' + str(zero_label_raman.shape[0]))
print('Sets with one label: ' + str(single_label_raman.shape[0]))
print('Sets with >1 labels: ' + str(mult_label_raman.shape[0]))
print('Mean set size:', prediction_sets_raman.sum(1).mean())
print('Std set size:', prediction_sets_raman.sum(1).std())
print('HQI cutt off:', 1-qhat_raman)

Sets with zero labels: 1419
Sets with one label: 294
Sets with >1 labels: 0
Mean set size: 0.17162872154115585
Std set size: 0.3770574273016038
HQI cutt off: 0.9577787447942816


In [34]:
output = [prediction_sets_n_raman[i][0] if i in single_label_raman else 
              '--outlier--' if i in zero_label_raman else 
              '--manual check!--' for i in range(label_test_raman.shape[0])]
output_df_raman = pd.DataFrame({'id':test_set_raman['id'],
                          'prection_set_raman':prediction_sets_n_raman,
                              #'output':output
                              })
output_df_raman.to_csv(directory_name+'/CP_Rubber_results_passive_unpaired.csv')
output_df_raman

,id,prection_set_raman
0,FHS_passive_July2024_D6_20240813_1,[]
1,FHS_passive_July2024_D6_20240813_2,[]
2,FHS_passive_July2024_D6_20240813_4,[]
3,FHS_passive_July2024_D6_20240813_5,[]
4,FHS_passive_July2024_D6_20240813_6,[]
...,...,...
1708,DC_passive_MarApr2024_C9_20240612_189,[]
1709,DC_passive_MarApr2024_C9_20240612_195,[]
1710,DC_passive_MarApr2024_C9_20240612_197,[]
1711,DC_passive_MarApr2024_C9_20240612_198,[Rubber]
